In [1]:
import os
import sys
os.chdir('..')
current_dir = os.path.abspath('')
sys.path.append(current_dir)

In [2]:
import scripts.helpers.helpers
import scripts.helpers.base_helpers
from pytorch_lightning import seed_everything
import importlib
from tqdm import tqdm
from scripts.demo.discretization import Img2ImgDiscretizationWrapper
from sgm.util import append_dims

In [3]:
importlib.reload(scripts.helpers.helpers)
importlib.reload(scripts.helpers.base_helpers)
from scripts.helpers.helpers import *
from scripts.helpers.base_helpers import *

In [4]:
seed_everything(42)
SAVE_PATH = "outputs/sample_giuded_interp_test/"
set_lowvram_mode(False)
model = load_model_from_joblib("./checkpoints/sdxlbase1_cache.joblib")

Global seed set to 42


In [5]:
options = {
  "discretization": "LegacyDDPMDiscretization",
  "sigma_min"     : 0.03,   #EDMDiscretization, [-inf, inf | 0.03]
  "sigma_max"     : 14.61,  #EDMDiscretization, [-inf, inf | 14.61]
  "rho"           : 3.0,    #EDMDiscretization, [-inf, inf | 3.0]

  "guider"                  : "IdentityGuider",
  "additional_guider_kwargs": {},
  "vanilla_cfg"             : 2.0,  #VanillaCFG, [0.0, inf | 5.0]
  "linear_cfg"              : 1.5,  #LinearCFG, [1.0, inf | 1.5]
  "triangle_cfg"            : 2.5,  #TriangleCFG, [1.0, 10.0 | 2.5]
  "min_cfg"                 : 1.0,  #LinearCFG TriangleCFG, [1.0]
  "num_frames"              : 25,   #LinearCFG TriangleCFG, [25]

  "sampler"           : "EulerEDMSampler",
  "s_churn"           : 0.0,    #EulerEDM HeunEDM [0.0, inf | 0.0]
  "s_tmin"            : 0.0,    #EulerEDM HeunEDM [0.0, inf |0.0]
  "s_tmax"            : 999.0,  #EulerEDM HeunEDM [0.0, inf | 999.0]
  "s_noise"           : 0.0,    #EulerEDM HeunEDM [0.0, inf | 0.0]
  "eta"               : 1.0,    #EulerAncestral DPMPP2SAncestral [0.0, inf | 1.0]
  "s_noise_ancestral" : 1.0,    #EulerAncestral DPMPP2SAncestral [0.0, inf | 1.0]
  "order"             : 4,      #LinearMultistep [1, inf | 4]

  "crop_coords_top"         : 0,    #[0, inf | 0]
  "crop_coords_left"        : 0,    #[0, inf | 0]
  "aesthetic_score"         : 6.0,  #[-inf, inf | 6.0]
  "negative_aesthetic_score": 2.5,  #[-inf, inf | 2.5]
  "fps"                     : 6,    #[1, inf | 6]
  "mb_id"                   : 127,  #[0, 511 | 127]
  "image_path"              : None
}

In [6]:
img = get_image(SAVE_PATH + "000000176.png")
transform = transforms.ToTensor()
img = transform(img).unsqueeze(0)
img = resize_to_div32(img)

In [18]:
prompts = ["A young man with a green jacket standing in a lush forest background, realistic, photograph, 4K"]
num_steps =  30
dims = (576, 1024) #list(img.shape[-2:])
sampler = init_sampling(options = options, steps = num_steps)

In [8]:
sampler.discretization = Img2ImgDiscretizationWrapper(
    sampler.discretization, strength=1.0
)

In [19]:
num_samples = [1]
batch = get_turbo_batch(prompts, dims)
batch_uc = copy.deepcopy(batch)
batch_uc['txt'] = ['' for _ in batch['txt']]
with torch.no_grad():
    with autocast("cuda"):
        with model.ema_scope():
            load_model(model.conditioner)
            c, uc = model.conditioner.get_unconditional_conditioning(batch, batch_uc, force_uc_zero_embeddings=["txt"])
            unload_model(model.conditioner)

In [20]:
with torch.no_grad():
    with autocast("cuda"):
        with model.ema_scope():
            load_model(model.first_stage_model)
            model.en_and_decode_n_samples_a_time = 1
            z = model.encode_first_stage(img.to("cuda"))
            unload_model(model.first_stage_model)

In [21]:
z.shape

torch.Size([1, 4, 136, 204])

In [22]:
noise = torch.randn_like(z)
sigmas = sampler.discretization(sampler.num_steps).cuda()
sigma = sigmas[0]

In [23]:
noised_z = z + noise * append_dims(sigma, z.ndim).cuda()
noised_z = noised_z / torch.sqrt(
    1.0 + sigmas[0] ** 2.0
)

In [24]:
reset_rng(42)
with torch.no_grad():
    with autocast("cuda"):
        with model.ema_scope():
            def denoiser(x, sigma, c):
                return model.denoiser(model.model, x, sigma, c)
            load_model(model.denoiser)
            load_model(model.model)
            samples_z = sampler(denoiser, noise, cond=c, uc=uc)
            unload_model(model.model)
            unload_model(model.denoiser)

##############################  Sampling setting  ##############################
Sampler: EulerEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: IdentityGuider


Sampling with EulerEDMSampler for 30 steps: 100%|██████████| 30/30 [00:07<00:00,  3.80it/s]


In [25]:
with torch.no_grad():
    with autocast("cuda"):
        with model.ema_scope():
            load_model(model.first_stage_model)
            model.en_and_decode_n_samples_a_time = 1
            samples_x = model.decode_first_stage(samples_z)
            unload_model(model.first_stage_model)
samples = torch.clamp((samples_x + 1.0) / 2.0, min=0.0, max=1.0)

In [26]:
save_png(SAVE_PATH, samples)

In [17]:
clear_vram()